# Day 13：Ollama + 本地模型部署

今天学怎么在本地跑 LLM。为什么要本地部署？

1. **隐私**：数据不出本机，适合处理敏感信息
2. **成本**：不需要付 API 费用，无限调用
3. **离线**：断网也能用
4. **面试**：证明你"真的动手做过"，不只是 API 调用者

Ollama 是目前最简单的本地 LLM 部署工具，一行命令就能跑起来。

## 一、GGUF 量化格式

本地部署的核心技术是**量化**——把大模型的权重压缩，减小体积和显存占用。

### 量化级别

| 级别 | 精度 | 7B 模型大小 | 质量 | 适用场景 |
|------|------|-----------|------|----------|
| FP16 | 16 bit | ~14 GB | 最佳 | 有充足显存时 |
| Q8_0 | 8 bit | ~7 GB | 接近 FP16 | 显存够但想省一点 |
| Q6_K | 6 bit | ~5.5 GB | 很好 | 平衡之选 |
| Q4_K_M | 4 bit | ~4 GB | 良好 | **推荐默认选择** |
| Q3_K_M | 3 bit | ~3 GB | 有损失 | 显存紧张时 |
| Q2_K | 2 bit | ~2.5 GB | 明显损失 | 只是试试 |

**Q4_K_M 是最佳平衡点**：模型大小减半，质量损失很小。

In [ ]:
# ===== 量化原理演示 =====
# 用简单例子理解量化的本质

import numpy as np

def quantize_demo():
    """
    量化的核心思想：
    把高精度数值映射到低精度的离散值
    """
    # 原始 FP16 权重（模拟）
    np.random.seed(42)
    weights_fp16 = np.random.randn(16).astype(np.float16)
    
    print("量化原理演示")
    print("=" * 50)
    
    # ===== INT8 量化 =====
    # 映射到 [-128, 127] 的整数
    scale_8 = np.max(np.abs(weights_fp16)) / 127
    weights_int8 = np.round(weights_fp16 / scale_8).astype(np.int8)
    weights_dequant_8 = weights_int8.astype(np.float16) * scale_8
    
    print("\nINT8 量化 (16bit → 8bit):")
    print(f"  原始值: {weights_fp16[:8]}")
    print(f"  量化值: {weights_int8[:8]}")
    print(f"  反量化: {weights_dequant_8[:8]}")
    print(f"  误差:   {np.abs(weights_fp16[:8] - weights_dequant_8[:8])}")
    
    # ===== INT4 量化 =====
    # 映射到 [-8, 7] 的整数
    scale_4 = np.max(np.abs(weights_fp16)) / 7
    weights_int4 = np.clip(np.round(weights_fp16 / scale_4), -8, 7).astype(np.int8)
    weights_dequant_4 = weights_int4.astype(np.float16) * scale_4
    
    print("\nINT4 量化 (16bit → 4bit):")
    print(f"  原始值: {weights_fp16[:8]}")
    print(f"  量化值: {weights_int4[:8]}")
    print(f"  反量化: {weights_dequant_4[:8]}")
    print(f"  误差:   {np.abs(weights_fp16[:8] - weights_dequant_4[:8])}")
    
    # 误差对比
    error_8 = np.mean(np.abs(weights_fp16 - weights_dequant_8))
    error_4 = np.mean(np.abs(weights_fp16 - weights_dequant_4))
    
    print("\n" + "=" * 50)
    print(f"平均误差: INT8={error_8:.6f}, INT4={error_4:.6f}")
    print(f"INT4 误差是 INT8 的 {error_4/error_8:.1f} 倍")
    print("\n结论：")
    print("- 量化是有损压缩，精度越高误差越小")
    print("- INT8 几乎无损，INT4 损失可控")
    print("- 实际模型中，大部分权重分布集中，量化效果比随机数据好得多")

quantize_demo()

## 二、GGUF 格式详解

GGUF（GPT-Generated Unified Format）是 llama.cpp 使用的模型格式。

```
GGML（旧格式）→ GGUF（新格式）
  - 支持更大的模型
  - 支持更多量化类型
  - 元数据更丰富
  - 社区标准格式
```

### Ollama 使用的模型来源

```
HuggingFace 上的原始模型
    ↓ 转换
GGUF 格式（llama.cpp 转换工具）
    ↓ 量化
Q4_K_M 等量化版本
    ↓ 上传
Ollama Registry（ollama.com/library）
    ↓ pull
本地运行
```

In [ ]:
# ===== Ollama 常用命令（2026 年） =====

ollama_commands = """
Ollama 常用命令速查（v0.22.x, 2026 年）
=" * 50

1. 模型管理
   ollama pull qwen3:14b                # Qwen3 14B（开源最强 Tool Calling）
   ollama pull deepseek-r1:14b          # DeepSeek-R1 推理模型
   ollama pull deepseek-v4-flash        # DeepSeek V4 Flash（1M 上下文，$0.07/M）
   ollama pull llama4:8b                # Llama 4 8B（Meta 最新）
   ollama pull gemma4:27b               # Gemma 4 27B（Google，Apache 2.0）
   ollama list                          # 查看已下载模型
   ollama rm qwen3:8b                   # 删除模型

2. 交互式对话
   ollama run qwen3:14b                 # 启动交互式对话
   ollama run deepseek-r1:14b "写一个快排"   # 单次推理

3. API 调用（OpenAI 兼容）
   curl http://localhost:11434/v1/chat/completions \\
     -H "Content-Type: application/json" \\
     -d '{"model": "deepseek-r1:14b", "messages": [{"role": "user", "content": "你好"}]}'

4. 模型信息
   ollama show qwen3:14b                # 查看模型详情
   ollama show qwen3:14b --modelfile    # 查看 Modelfile

5. 2026 年推荐模型（本地部署）
   - Qwen3 14B (qwen3:14b) — 阿里开源旗舰，119 语言，Tool Calling 最强
   - DeepSeek-R1 14B (deepseek-r1:14b) — 推理链模型，数学/代码优秀
   - DeepSeek V4 Flash — 1M 长上下文，MIT 协议，极致性价比
   - Llama 4 Scout (llama4) — 10M 超长上下文窗口
   - Gemma 4 27B (gemma4:27b) — Google 开源，Apache 2.0
   - Kimi K2.6 — SWE-bench Pro 58.6%（#1 开源编程模型）

6. 硬件建议（Q4_K_M 量化）
   - 8GB 显存：Qwen3 8B / DeepSeek-R1 7B / Llama 4 8B
   - 16GB 显存：Qwen3 14B / DeepSeek-R1 14B
   - 24GB 显存：Qwen3 32B / DeepSeek-R1 32B
   - 32GB+ 显存：Llama 4 70B / DeepSeek V4 Flash
"""

print(ollama_commands)

In [ ]:
# ===== 用 Python 调用 Ollama API =====
# Ollama 提供 OpenAI 兼容的 API，可以直接用 OpenAI SDK 调用

print("方式1: 使用 OpenAI SDK（推荐）")
print("=" * 50)
print("""
from openai import OpenAI

# 指向 Ollama 的本地端点
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="none",  # Ollama 不需要 API Key
)

response = client.chat.completions.create(
    model="qwen3:8b",  # 2025 年推荐模型
    messages=[
        {"role": "system", "content": "你是一个有帮助的助手"},
        {"role": "user", "content": "什么是 KV Cache？"}
    ],
    temperature=0.7,
    max_tokens=500,
)

print(response.choices[0].message.content)
""")

print("\n方式2: 使用 requests 库")
print("=" * 50)
print("""
import requests

response = requests.post(
    "http://localhost:11434/v1/chat/completions",
    json={
        "model": "qwen3:8b",
        "messages": [{"role": "user", "content": "你好"}],
    }
)

print(response.json()["choices"][0]["message"]["content"])
""")

print("\n方式3: 使用 LangChain（和之前一样）")
print("=" * 50)
print("""
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen3:8b",
    base_url="http://localhost:11434/v1",
    api_key="none",
)

response = llm.invoke("你好")
print(response.content)
""")

## 三、量化对推理速度的影响

量化不仅减小模型体积，还影响推理速度。

### 速度分析

| 量化级别 | 模型大小 | 推理速度 | 原因 |
|---------|---------|---------|------|
| FP16 | 14 GB | 基准 | 数据量大，显存带宽瓶颈 |
| Q8_0 | 7 GB | 快 1.5-2x | 数据量减半，带宽压力小 |
| Q4_K_M | 4 GB | 快 2-3x | 数据量更小，CPU 也能跑 |
| Q2_K | 2.5 GB | 快但质量差 | 精度损失太大，不推荐 |

**关键洞察**：量化后模型更小，每步需要读取的数据量更少，
在 Memory-bound 的 Decode 阶段，速度提升明显。

In [ ]:
# ===== 不同量化级别的性能分析 =====

def analyze_quantization_impact(
    model_params_b: float,
    gpu_bandwidth_gbs: float,
    cpu_bandwidth_gbs: float = 50,  # CPU 内存带宽
):
    """
    分析不同量化级别的推理速度
    
    Decode 阶段每步需要读取的数据量 ≈ 2 * params * dtype_bytes
    推理时间 ≈ 数据量 / 带宽
    """
    configs = [
        ("FP16", 2.0, 14.0),
        ("Q8_0", 1.0, 7.0),
        ("Q6_K", 0.75, 5.5),
        ("Q4_K_M", 0.5, 4.0),
        ("Q3_K_M", 0.375, 3.0),
        ("Q2_K", 0.25, 2.5),
    ]
    
    print(f"\n{model_params_b}B 模型量化性能分析")
    print("=" * 65)
    print(f"{'量化':<10} {'大小(GB)':<10} {'GPU速度(tok/s)':<15} {'CPU速度(tok/s)':<15} {'质量'}")
    print("-" * 65)
    
    fp16_gpu_speed = None
    
    for name, dtype_bytes, model_size_gb in configs:
        # 每步数据量
        data_per_step_gb = 2 * model_params_b * 1e9 * dtype_bytes / 1e9
        
        # GPU 速度
        gpu_time_ms = data_per_step_gb / gpu_bandwidth_gbs * 1000
        gpu_speed = 1000 / gpu_time_ms  # tokens/s
        
        # CPU 速度
        cpu_time_ms = data_per_step_gb / cpu_bandwidth_gbs * 1000
        cpu_speed = 1000 / cpu_time_ms
        
        if fp16_gpu_speed is None:
            fp16_gpu_speed = gpu_speed
        
        quality = {"FP16": "最佳", "Q8_0": "接近FP16", "Q6_K": "很好",
                   "Q4_K_M": "良好", "Q3_K_M": "有损失", "Q2_K": "明显损失"}
        
        print(f"{name:<10} {model_size_gb:<10.1f} {gpu_speed:<15.1f} {cpu_speed:<15.1f} {quality[name]}")
    
    print("\n" + "=" * 65)
    print("建议：")
    print("  - 有 GPU：用 Q4_K_M 或 Q8_0，速度和质量都好")
    print("  - 纯 CPU：用 Q4_K_M，CPU 也能流畅运行 7B 模型")
    print("  - 显存不足：用 Q3_K_M，但质量有明显下降")

# 不同硬件的分析
print("不同硬件下的量化性能（7B 模型）")
analyze_quantization_impact(7, gpu_bandwidth_gbs=1008)  # RTX 4090
analyze_quantization_impact(7, gpu_bandwidth_gbs=50)    # CPU only

## 四、本地部署的实际意义

### 4.1 什么时候用本地模型

| 场景 | 推荐方案 | 原因 |
|------|---------|------|
| 生产环境高并发 | vLLM + API | 需要稳定性和速度 |
| 开发调试 | Ollama 本地 | 免费、快速迭代 |
| 敏感数据处理 | 本地模型 | 数据不出本机 |
| 离线环境 | 本地模型 | 无网络依赖 |
| 预算有限 | 本地模型 | 无 API 费用 |
| 复杂推理任务 | Claude/GPT-4 | 本地模型能力不够 |

### 4.2 本地模型的局限

- 7B 模型的推理能力远不如 Claude/GPT-4
- Tool Calling 的格式准确率较低
- 复杂指令遵循能力弱
- 适合简单任务：文本提取、格式转换、简单问答

In [ ]:
# ===== 把 RAG 系统的 LLM 替换成本地模型（2026 年） =====

print("RAG 系统接入本地模型的代码示例")
print("=" * 50)
print("""
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# 切换到本地 Ollama 模型（2026 年推荐 Qwen3 14B 或 DeepSeek-R1 14B）
llm = ChatOpenAI(
    model="qwen3:14b",  # 2026 年开源 Tool Calling 最强
    base_url="http://localhost:11434/v1",
    api_key="none",
    temperature=0,
)

def rag_query(question: str, context: str) -> str:
    messages = [
        SystemMessage(content="基于以下上下文回答问题。如果上下文中没有相关信息，请说明。"),
        HumanMessage(content=f"上下文：{context}\\n\\n问题：{question}")
    ]
    response = llm.invoke(messages)
    return response.content

# 使用示例
context = "LangGraph 是 LangChain 团队开发的状态图框架..."
answer = rag_query("什么是 LangGraph？", context)
print(answer)
""")

print("\n" + "=" * 50)
print("\n2026 年切换本地/远程模型只需改 base_url 和 model：")
print("  - 本地 Ollama:    http://localhost:11434/v1  (model: qwen3:14b)")
print("  - 本地 vLLM:      http://localhost:8000/v1   (model: Qwen/Qwen3-14B)")
print("  - DeepSeek:       https://api.deepseek.com   (model: deepseek-chat)")
print("  - Kimi (Moonshot): https://api.moonshot.cn   (model: kimi-k2.6)")
print("  - OpenAI:         https://api.openai.com/v1  (model: gpt-5.5)")
print("\n代码完全一样，只改连接地址。OpenAI 兼容 API 是 2026 年的行业标准。")

In [ ]:
# ===== 不同推理方案的成本对比（2026 年） =====

print("不同推理方案成本对比（100 万 Token/月，2026 年 Q2 参考价）")
print("=" * 75)

solutions = [
    {
        "name": "Claude Opus 4.7",
        "monthly_cost": "~$15",
        "quality": "最强（SWE-bench 87.6%）",
        "context": "1M",
        "note": "复杂推理/多文件代码"
    },
    {
        "name": "GPT-5.5 Pro",
        "monthly_cost": "~$18",
        "quality": "最强（并行推理）",
        "context": "1M",
        "note": "Terminal-Bench #1"
    },
    {
        "name": "DeepSeek V4-Pro",
        "monthly_cost": "~$0.5",
        "quality": "前沿（SWE-bench 80.6%）",
        "context": "1M",
        "note": "性价比之王，MIT 协议"
    },
    {
        "name": "DeepSeek V4-Flash",
        "monthly_cost": "~$0.04",
        "quality": "优秀",
        "context": "1M",
        "note": "极致低价，$0.07/M output"
    },
    {
        "name": "Kimi K2.6 (API)",
        "monthly_cost": "~$1.2",
        "quality": "编码 #1（SWE-bench Pro 58.6%）",
        "context": "256K",
        "note": "MIT 协议，开源编程最强"
    },
    {
        "name": "Ollama 本地 (Qwen3 14B)",
        "monthly_cost": "电费 ~$5/月",
        "quality": "良好（Tool Calling 优秀）",
        "context": "32K",
        "note": "数据不出本机"
    },
]

print(f"{'方案':<25} {'月成本':<15} {'质量':<30} {'上下文':<10}")
print("-" * 80)
for s in solutions:
    print(f"{s['name']:<25} {s['monthly_cost']:<15} {s['quality']:<30} {s['context']:<10}")
    print(f"  → {s['note']}")

print("\n" + "=" * 75)
print("\n2026 年选择建议：")
print("  - 日常开发（性价比）：DeepSeek V4-Flash/Pro — 质量够好，价格是 GPT-5.5 的 1/30")
print("  - 编码任务（开源最强）：Kimi K2.6 — SWE-bench Pro 58.6%，MIT 协议")
print("  - 复杂推理（质量优先）：Claude Opus 4.7 — SWE-bench 87.6%")
print("  - 敏感数据：Ollama 本地 Qwen3 14B — Apache 2.0，无数据外泄")
print("  - 混合策略：80% DeepSeek V4 + 15% Kimi K2.6 + 5% Claude Opus 4.7（总成本降 60%）")

## 今日总结

今天掌握了本地模型部署的核心知识。

| 概念 | 关键点 |
|------|--------|
| GGUF | llama.cpp 的模型格式，社区标准 |
| 量化 | FP16→INT4，模型大小减半，质量损失小 |
| Q4_K_M | 最佳平衡点，推荐默认选择 |
| Ollama | 最简单的本地部署工具，一行命令启动 |
| OpenAI 兼容 | 本地/远程切换只改 base_url |

**明天学什么：** Week 2 整合——把 RAG、Rerank、本地模型组合成完整系统。

**写在简历上的要点：**
"熟悉 LLM 本地部署方案，能使用 Ollama/vLLM 部署量化模型（GGUF Q4_K_M），理解量化原理（INT8/INT4 量化、精度-大小权衡），能根据场景选择合适的部署方案。"